# 55. `YieldAsymmetry`: a raw counting CP asymmetry

**Objectives:**

- Fit an extended joint CP likelihood with the default, shared `signal_yield` (amplitude-driven
  charge split via the jointly-normalized `I_plus`/`I_minus`).
- Rebuild the same fit with `YieldAsymmetry`, splitting a total `N_s` into independently
  fittable `N_plus`/`N_minus` through a raw counting asymmetry `A_yield`.
- Discuss when each parameterization is the right one.

Run the cells in order in a fresh kernel. Masses are in GeV, invariants in GeV^2. See
`docs/cp_coefficients.md`'s "Yield-asymmetry parameterization" section for the full
convention.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede any numerical work: amplitudes use complex128.

import matplotlib.pyplot as plt

from dalitzplotfitter import (
    CPFitSession, CPRealImag, DecayChannel, DecayModel, NonResonant, Parameter,
    RealImag, Resonance, YieldAsymmetry, generate_cp_toy,
)

## 1. A small B -> K pi pi CP model

One fixed `Kstar` reference resonance plus a `NonResonant` term with a `CPRealImag`
coefficient (`c_q = (x + q dx) + i (y + q dy)`, `q = +-1`), shared between both charges so the
session collects each parameter once (as in tutorial 6).

In [2]:
cp = CPRealImag(*[
    Parameter.coefficient(f"NR.{name}", value, owner="NR", bounds=bounds, step=0.02)
    for name, value, bounds in [
        ("x", 0.8, (-2, 2)), ("y", 0.3, (-2, 2)),
        ("dx", 0.10, (-0.4, 0.4)), ("dy", -0.08, (-0.4, 0.4)),
    ]
])

def make_model(parent, daughters, charge):
    return DecayModel(
        DecayChannel(parent, daughters),
        [Resonance("Kstar", (0, 2), RealImag(1.0, 0.0),
                   mass=0.8958, width=0.0474, spin=1),
         NonResonant(cp.for_charge(charge), name="NR")],
        normalization_method="square-dalitz", normalization_pair=(0, 2),
        normalization_resolution=80,
    )

plus_model = make_model("B+", ("K+", "pi+", "pi-"), +1)
minus_model = make_model("B-", ("K-", "pi-", "pi+"), -1)
truth = {p.name: p.value for p in plus_model.parameters}
plus_data, minus_data = generate_cp_toy(
    plus_model, minus_model, 4000, parameters=truth, seed=55,
    inverse_resolution=400, include_momenta=False,
)
print("Charge counts:", plus_data.size, minus_data.size)

Charge counts: 2124 1876


## 2. Extended fit with a plain shared `signal_yield`

By default, one `Parameter` multiplies both charges' *jointly*-normalized isobar PDFs
(`p_q(phi) = |A_q(phi)|^2 / (I_plus + I_minus)`), so the charge split comes entirely from the
amplitude's own `I_plus`/`I_minus` -- exactly `CPJointNLL`'s "central invariant" joint
normalization (`docs/cp_coefficients.md`).

In [3]:
n_total = plus_data.size + minus_data.size
shared_yield = Parameter("n_signal", float(n_total), bounds=(0.0, 4.0 * n_total))

shared_session = CPFitSession(
    plus_model, minus_model, plus_data, minus_data,
    extended=True, signal_yield=shared_yield,
)
start = {name: value + 0.03 for name, value in truth.items()}
start["n_signal"] = float(n_total) * 0.9
shared_result = shared_session.fit(start, simplex=True, ncall=8000)
shared_session.report(shared_result)
assert shared_result.valid
print(f"Fitted N_sig={shared_result.values['n_signal']:.1f} (generated total={n_total})")

valid=True  NLL=-9342.196616
parameter                           value            error
NR.x                             0.850252        0.0211596
NR.y                             0.300203        0.0466457
NR.dx                            0.102102        0.0187154
NR.dy                          -0.0746959        0.0464654
n_signal                             4000          63.2448


predicted charge fractions: B+=0.535203  B-=0.464797
B+ fit fractions


Fit fractions (physical)
component                    fraction [%]
Kstar                              51.077
NR                                 48.923
sum                               100.000

B- fit fractions


Fit fractions (physical)
component                    fraction [%]
Kstar                              58.814
NR                                 41.186
sum                               100.000
Fitted N_sig=4000.0 (generated total=4000)


## 3. Extended fit with `YieldAsymmetry`

`YieldAsymmetry(total, asymmetry)` replaces the shared `signal_yield` with independently
fittable literal per-charge counts, `N_plus = N_s (1 - A_yield) / 2`,
`N_minus = N_s (1 + A_yield) / 2`. Each charge's isobar PDF is then normalized *on its own*
(`|A_q|^2 / I_q`) rather than jointly, so `A_yield` sets the relative rate between charges
directly instead of leaving it to `I_plus`/`I_minus`.

In [4]:
yield_asymmetry = YieldAsymmetry(
    Parameter("N_s", float(n_total), bounds=(0.0, 4.0 * n_total)),
    Parameter("A_yield", 0.0, bounds=(-1.0, 1.0)),
)

asym_session = CPFitSession(
    plus_model, minus_model, plus_data, minus_data,
    extended=True, signal_yield=yield_asymmetry,
)
start_asym = dict(start)
del start_asym["n_signal"]
start_asym["N_s"] = float(n_total) * 0.9
start_asym["A_yield"] = 0.05
asym_result = asym_session.fit(start_asym, simplex=True, ncall=8000)
asym_session.report(asym_result)
assert asym_result.valid

n_plus_fit = yield_asymmetry.plus(asym_session.result_values(asym_result))
n_minus_fit = yield_asymmetry.minus(asym_session.result_values(asym_result))
print(f"Fitted N_s={asym_result.values['N_s']:.1f}, A_yield={asym_result.values['A_yield']:.4f}")
print(f"-> N_plus={float(n_plus_fit):.1f}, N_minus={float(n_minus_fit):.1f} "
      f"(generated: {plus_data.size}, {minus_data.size})")

valid=True  NLL=-9342.467450
parameter                           value            error
NR.x                             0.851071        0.0211437
NR.y                             0.299195        0.0467404
NR.dx                            0.109403        0.0211385
NR.dy                          -0.0724358        0.0466519
N_s                                  4000          63.2445
A_yield                            -0.062        0.0157803
predicted charge fractions: B+=0.531000  B-=0.469000
B+ fit fractions
Fit fractions (physical)
component                    fraction [%]
Kstar                              50.660
NR                                 49.340
sum                               100.000

B- fit fractions
Fit fractions (physical)
component                    fraction [%]
Kstar                              59.235
NR                                 40.765
sum                               100.000
Fitted N_s=4000.0, A_yield=-0.0620
-> N_plus=2124.0, N_minus=1876.0 (generated: 2124

## 4. When to use which

- **Shared `signal_yield` (default)**: the charge split is derived entirely from the coherent
  amplitude's interference, `I_plus` vs `I_minus`. This is the parameterization that gives the
  fit sensitivity to the *interference-driven* CP asymmetry -- the physically interesting
  quantity in most amplitude analyses.
- **`YieldAsymmetry`**: use it when the observable of interest is instead a *raw/counting*
  asymmetry between B+ and B- candidates -- e.g. one contaminated by production or detection
  asymmetries that the amplitude model does not (and should not try to) capture. `A_yield` is
  unrelated to `CPRealImag`'s coefficient-level `A_CP`; the two are independent observables and
  should not be conflated when naming or interpreting fit parameters.

Both fits above recover consistent per-charge event counts, as expected: the toy was generated
with no injected yield asymmetry, so `A_yield` fits close to zero and the two parameterizations
agree on `N_plus`/`N_minus` even though they reach it through different mechanisms.

## Continue learning

See [`docs/cp_coefficients.md`](../../docs/cp_coefficients.md), "Yield-asymmetry
parameterization", and [tutorial 6](tutorial_06_joint_cp_fit.ipynb) for the plain joint CP fit
this notebook builds on.

Return to [the course guide](TUTORIALS.md).